# 26 · Advanced Capstone — Analytics Engineering

The final boss. 🐉 These challenges combine window functions, CTEs, JSON,
conditional aggregation, and the recipes from module 25. Attempt each before
revealing the solution.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

### Challenge 1 — Month-over-month growth
*Skills: dates, window functions, LAG.*

Compute completed-order revenue per month and the percentage change vs the previous month.

**✏️ Exercise 1.** Monthly revenue with month-over-month % change.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH m AS (
  SELECT STRFTIME('%Y-%m', o.order_date) AS month,
         SUM(oi.quantity * oi.unit_price) AS revenue
  FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
  WHERE o.status = 'completed'
  GROUP BY month
)
SELECT month, ROUND(revenue,2) AS revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month))
             / LAG(revenue) OVER (ORDER BY month), 1) AS mom_pct
FROM m ORDER BY month;

### Challenge 2 — Customer RFM-style summary
*Skills: joins, aggregation, dates.*

For each customer: number of completed orders (frequency), total spend (monetary), and days since their last order as of 2024-08-01 (recency).

**✏️ Exercise 2.** Recency / frequency / monetary summary per customer.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT cu.first_name || ' ' || cu.last_name AS customer,
       COUNT(DISTINCT o.order_id) AS frequency,
       ROUND(COALESCE(SUM(oi.quantity * oi.unit_price), 0), 2) AS monetary,
       CAST(JULIANDAY('2024-08-01') - JULIANDAY(MAX(o.order_date)) AS INTEGER) AS recency_days
FROM customers cu
LEFT JOIN orders o      ON o.customer_id = cu.customer_id AND o.status = 'completed'
LEFT JOIN order_items oi ON oi.order_id = o.order_id
GROUP BY cu.customer_id
ORDER BY monetary DESC;

### Challenge 3 — Top product per category with its share
*Skills: window functions, partitioned share.*

For each category, show its best-selling product (by units) and what % of the category's units that product represents.

**✏️ Exercise 3.** Best product per category + its share of category units.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH sales AS (
  SELECT p.category_id, p.product_name, SUM(oi.quantity) AS units
  FROM order_items oi JOIN products p ON oi.product_id = p.product_id
  GROUP BY p.category_id, p.product_name
),
annotated AS (
  SELECT category_id, product_name, units,
         ROW_NUMBER() OVER (PARTITION BY category_id ORDER BY units DESC) AS rn,
         ROUND(100.0 * units / SUM(units) OVER (PARTITION BY category_id), 1) AS pct_of_cat
  FROM sales
)
SELECT c.category_name, a.product_name, a.units, a.pct_of_cat
FROM annotated a JOIN categories c ON a.category_id = c.category_id
WHERE a.rn = 1 ORDER BY c.category_name;

### Challenge 4 — Build a JSON API payload
*Skills: JSON building, joins, aggregation.*

Produce one JSON object per category containing the category name, product count, and an array of `{name, price}` objects for its products.

**✏️ Exercise 4.** One JSON document per category with nested products.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT json_object(
         'category', c.category_name,
         'product_count', COUNT(*),
         'products', json_group_array(json_object('name', p.product_name, 'price', p.unit_price))
       ) AS category_payload
FROM categories c
JOIN products p ON p.category_id = c.category_id
GROUP BY c.category_name
ORDER BY c.category_name;

### Challenge 5 — Employee performance ranking
*Skills: joins, aggregation, self-join, window ranking.*

Rank sales reps by completed revenue, showing their manager and dense rank.

**✏️ Exercise 5.** Rep leaderboard with manager and dense rank.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
WITH perf AS (
  SELECT e.employee_id, e.first_name || ' ' || e.last_name AS employee, e.manager_id,
         COALESCE(SUM(oi.quantity * oi.unit_price), 0) AS revenue
  FROM employees e
  LEFT JOIN orders o      ON o.employee_id = e.employee_id AND o.status = 'completed'
  LEFT JOIN order_items oi ON oi.order_id = o.order_id
  GROUP BY e.employee_id
)
SELECT perf.employee,
       m.first_name || ' ' || m.last_name AS manager,
       ROUND(perf.revenue, 2) AS revenue,
       DENSE_RANK() OVER (ORDER BY perf.revenue DESC) AS rank
FROM perf
LEFT JOIN employees m ON perf.manager_id = m.employee_id
ORDER BY rank;

## 🏆 You've completed the full SQL Zero-to-Hero Bootcamp — Advanced Track included!

You now command the entire toolkit: set-based thinking, every join type, advanced
aggregation and window functions, NULL-safe logic, schema design and
normalization, indexing and performance tuning, JSON, triggers, and the query
patterns that solve real problems.

**Keep growing:** apply these to your own datasets, learn your production
database's dialect (PostgreSQL / MySQL / SQL Server differ mainly at the edges),
and practice reading `EXPLAIN` plans on real tables. You're now a strong SQL
engineer. 🚀